# Avaliação walk-forward final

Este notebook será o orquestrador da avaliação comparativa final. Ele só será habilitado quando o horizonte e a disponibilidade das variáveis externas estiverem registrados.

O bloco de parâmetros abaixo permite verificar as datas e o número de origens de um horizonte candidato, sem executar modelos finais ou calcular MAE.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
config = json.loads((ROOT / 'configs' / 'base_experiment_rules.json').read_text(encoding='utf-8'))
sys.path.insert(0, str(ROOT / 'src'))

# Altere somente depois da aprovação do grupo/professor. Uma etapa equivale
# à granularidade da base: dia, hora ou semana encerrada na sexta-feira.
HORIZON_STEPS_BY_BASE = {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}
HORIZON_APPROVED = False

print(config['status'])
print(f'Horizontes candidatos: {HORIZON_STEPS_BY_BASE}')
print('Resultado: não executar MAE ou ranking final enquanto HORIZON_APPROVED for False.')

granularity, random state and train-test fractions confirmed by the group; final forecast horizon remains pending
Horizontes candidatos: {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}
Resultado: não executar MAE ou ranking final enquanto HORIZON_APPROVED for False.


In [2]:
import pandas as pd

from discovery_series_temporais.series import build_origin_schedule, load_candidate_target

preview = []
for base_id, horizon_steps in HORIZON_STEPS_BY_BASE.items():
    candidate = load_candidate_target(base_id, ROOT / 'data' / 'processed')
    schedule = build_origin_schedule(candidate, horizon_steps=horizon_steps)
    preview.append({
        'base': base_id,
        'horizon_steps': horizon_steps,
        'frequency': candidate.frequency_label,
        'test_origins': len(schedule),
        'first_origin': schedule['origin_timestamp'].min(),
        'first_target': schedule['target_timestamp'].min(),
    })

display(pd.DataFrame(preview))
if not HORIZON_APPROVED:
    print('Prévia somente: escolha e aprove o horizonte antes de habilitar a avaliação final.')

,base,horizon_steps,frequency,test_origins,first_origin,first_target
0,1,1,daily,851,2022-06-08 00:00:00,2022-06-09 00:00:00
1,2,1,hourly (target-only consolidation),7598,2017-11-16 20:00:00,2017-11-16 21:00:00
2,3,1,hourly,6786,2016-05-15 21:00:00,2016-05-15 22:00:00
3,4,1,hourly mean,14008,2015-05-25 06:00:00,2015-05-25 07:00:00
4,5,1,weekly ending Friday using last observed price,481,2002-10-11 00:00:00,2002-10-18 00:00:00


Prévia somente: escolha e aprove o horizonte antes de habilitar a avaliação final.


## Sequência que este notebook executará depois da liberação

1. Carregar a matriz de features aprovada e formar a máscara comum de elegibilidade.
2. Gerar a agenda final uma única vez por base.
3. Ajustar os hiperparâmetros dentro do treino com walk-forward expansivo.
4. Reestimar os quatro modelos em cada origem final.
5. Validar as agendas antes do MAE, salvar resíduos e produzir as tabelas comparativas.